# ML Revenue Regression — Financial Services Demo
Train and evaluate regression models to forecast monthly revenue using features from `CONSUMPTION.DT_MONTHLY_REVENUE`.

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark import Session
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# Cell 2: Connect to Snowflake
connection_params = {
    'account':   os.environ.get('SNOWFLAKE_ACCOUNT', 'your_account'),
    'user':      os.environ.get('SNOWFLAKE_USER', 'your_user'),
    'password':  os.environ.get('SNOWFLAKE_PASSWORD', ''),
    'role':      'ACCOUNTADMIN',
    'warehouse': 'FINSERV_WH',
    'database':  'FINSERV_DB',
    'schema':    'CONSUMPTION'
}

session = Session.builder.configs(connection_params).create()
print(f'Connected: {session.get_current_database()}.{session.get_current_schema()}')

In [ ]:
# Cell 3: Load monthly revenue data
df_sp = session.table('DT_MONTHLY_REVENUE')
print(f'Schema: {df_sp.schema}')
print(f'Row count: {df_sp.count():,}')

df = df_sp.to_pandas()
df = df.sort_values('REVENUE_MONTH')
df.head(10)

In [ ]:
# Cell 4: EDA — Revenue trend
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].plot(df['REVENUE_MONTH'], df['TOTAL_REVENUE'], marker='o', markersize=3)
axes[0,0].set_title('Monthly Revenue Trend')
axes[0,0].set_xlabel('Month')
axes[0,0].set_ylabel('Revenue ($)')
axes[0,0].tick_params(axis='x', rotation=45)

axes[0,1].plot(df['REVENUE_MONTH'], df['TRANSACTION_COUNT'], marker='s', markersize=3, color='coral')
axes[0,1].set_title('Monthly Transaction Count')
axes[0,1].tick_params(axis='x', rotation=45)

if 'UNIQUE_CUSTOMERS' in df.columns:
    axes[1,0].plot(df['REVENUE_MONTH'], df['UNIQUE_CUSTOMERS'], marker='^', markersize=3, color='green')
    axes[1,0].set_title('Unique Customers per Month')
    axes[1,0].tick_params(axis='x', rotation=45)

if 'AVG_TRANSACTION' in df.columns:
    axes[1,1].plot(df['REVENUE_MONTH'], df['AVG_TRANSACTION'], marker='d', markersize=3, color='purple')
    axes[1,1].set_title('Average Transaction Amount')
    axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Feature engineering — lag features and rolling averages
df_feat = df.copy()

# Sort by month
df_feat = df_feat.sort_values('REVENUE_MONTH').reset_index(drop=True)

# Lag features
for lag in [1, 2, 3]:
    df_feat[f'REVENUE_LAG_{lag}'] = df_feat['TOTAL_REVENUE'].shift(lag)
    df_feat[f'TXN_COUNT_LAG_{lag}'] = df_feat['TRANSACTION_COUNT'].shift(lag)

# Rolling averages
df_feat['REVENUE_ROLL_3'] = df_feat['TOTAL_REVENUE'].rolling(3).mean()
df_feat['REVENUE_ROLL_6'] = df_feat['TOTAL_REVENUE'].rolling(6).mean()
df_feat['TXN_ROLL_3'] = df_feat['TRANSACTION_COUNT'].rolling(3).mean()

# Month-over-month growth
df_feat['REVENUE_GROWTH'] = df_feat['TOTAL_REVENUE'].pct_change()

# Drop rows with NaN from lags
df_feat = df_feat.dropna()
print(f'Rows after feature engineering: {len(df_feat)}')
df_feat.head()

In [ ]:
# Cell 6: Define features and target
exclude_cols = ['REVENUE_MONTH', 'TOTAL_REVENUE']
feature_cols = [c for c in df_feat.select_dtypes(include=[np.number]).columns
                if c not in exclude_cols]
target_col = 'TOTAL_REVENUE'

print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'Target: {target_col}')

X = df_feat[feature_cols]
y = df_feat[target_col]
print(f'X shape: {X.shape}')

In [ ]:
# Cell 7: Train/test split (time-based)
from sklearn.preprocessing import StandardScaler

# Use last 20% as test (time-ordered, no shuffle)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} months')
print(f'Test:  {X_test.shape[0]} months')

In [ ]:
# Cell 8: Train 5 regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    'LinearRegression':     LinearRegression(),
    'Ridge':                Ridge(alpha=1.0),
    'Lasso':                Lasso(alpha=1.0, max_iter=5000),
    'RandomForest':         RandomForestRegressor(n_estimators=100, random_state=42),
    'GradientBoosting':     GradientBoostingRegressor(n_estimators=100, random_state=42),
}

trained = {}
for name, model in models.items():
    use_scaled = name in ('LinearRegression', 'Ridge', 'Lasso')
    data = X_train_scaled if use_scaled else X_train
    model.fit(data, y_train)
    trained[name] = model
    print(f'{name} trained.')

In [ ]:
# Cell 9: Evaluate models
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []
for name, model in trained.items():
    use_scaled = name in ('LinearRegression', 'Ridge', 'Lasso')
    data = X_test_scaled if use_scaled else X_test
    y_pred = model.predict(data)

    metrics = {
        'Model': name,
        'MAE':   mean_absolute_error(y_test, y_pred),
        'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2':    r2_score(y_test, y_pred),
        'MAPE':  np.mean(np.abs((y_test - y_pred) / y_test)) * 100 if (y_test != 0).all() else np.nan,
    }
    results.append(metrics)
    print(f'{name}: R2={metrics["R2"]:.4f}, MAE={metrics["MAE"]:,.0f}, RMSE={metrics["RMSE"]:,.0f}')

df_results = pd.DataFrame(results)
df_results

In [ ]:
# Cell 10: Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Best model by R2
best_idx = df_results['R2'].idxmax()
best_name = df_results.loc[best_idx, 'Model']
best_model = trained[best_name]

use_scaled = best_name in ('LinearRegression', 'Ridge', 'Lasso')
data = X_test_scaled if use_scaled else X_test
y_pred = best_model.predict(data)

# Scatter: actual vs predicted
axes[0].scatter(y_test, y_pred, alpha=0.7, color='steelblue')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Revenue')
axes[0].set_ylabel('Predicted Revenue')
axes[0].set_title(f'Actual vs Predicted — {best_name}')

# Time series: actual vs predicted
test_months = df_feat['REVENUE_MONTH'].iloc[split_idx:].values
axes[1].plot(test_months, y_test.values, 'b-o', label='Actual', markersize=4)
axes[1].plot(test_months, y_pred, 'r--s', label='Predicted', markersize=4)
axes[1].set_title(f'Revenue Forecast — {best_name}')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Revenue ($)')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Residual analysis
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred, residuals, alpha=0.7, color='steelblue')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Revenue')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=15, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Feature importance (best tree model)
tree_models = {n: m for n, m in trained.items() if hasattr(m, 'feature_importances_')}
if tree_models:
    best_tree_name = max(tree_models, key=lambda n: df_results.loc[df_results['Model']==n, 'R2'].values[0])
    importances = tree_models[best_tree_name].feature_importances_
    feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
    feat_imp = feat_imp.sort_values('Importance', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='coral')
    ax.set_title(f'Feature Importance — {best_tree_name}')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 13: Register best model to Snowflake Model Registry
from snowflake.ml.registry import Registry

print(f'Best model: {best_name} (R2={df_results.loc[best_idx, "R2"]:.4f})')

registry = Registry(session=session)
mv = registry.log_model(
    model=best_model,
    model_name='REVENUE_FORECASTER',
    version_name='v1',
    sample_input_data=X_train.head(10),
    comment=f'Revenue regression — {best_name}'
)
print(f'Registered: REVENUE_FORECASTER v1')

In [ ]:
# Cell 14: Test inference from registry
mv_loaded = registry.get_model('REVENUE_FORECASTER').version('v1')
predictions = mv_loaded.run(X_test.head(5), function_name='predict')
print('Sample predictions from registry:')
predictions

In [ ]:
# Cell 15: Model comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df_results, x='Model', y='R2', ax=axes[0], palette='viridis')
axes[0].set_title('R-Squared by Model')
axes[0].set_ylim(min(0, df_results['R2'].min() - 0.1), 1.05)
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(data=df_results, x='Model', y='RMSE', ax=axes[1], palette='magma')
axes[1].set_title('RMSE by Model')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: Cleanup
session.close()
print('Session closed. Revenue regression complete.')